# Stats for the Verb Phrase

In [2]:
#first parse the corpus
# Import the toolkit, giving it a shorter alias 'mptf'
import mptf_parser as mptf

# Set your folder paths
input_folder = r"C:\Users\rahaa\Dropbox\MPCD\exports_28-1-2026"
output_folder = r"C:\Users\rahaa\Dropbox\MPCD\the syntax project\nounphrase\export_files\conllu_output"

In [7]:
import os
from conllu import parse
from conllu.parser import DEFAULT_FIELD_PARSERS
import mptf_parser as mptf

# ======================================================
# LOAD BOTH CORPORA
# ======================================================

print("\n--- Loading corpora ---")

custom_field_parsers = DEFAULT_FIELD_PARSERS.copy()
custom_field_parsers["id"] = lambda line, i: line[i]
custom_field_parsers["head"] = lambda line, i: line[i]

my_corpus = []
syntactically_annotated_corpus = []

conllu_files = sorted(
    f for f in os.listdir(output_folder)
    if f.lower().endswith(".conllu")
    and "outdated" not in f.lower()
)

for filename in conllu_files:
    file_path = os.path.join(output_folder, filename)

    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = f.read()

    # --- Clean malformed lines ---
    lines = raw_data.splitlines()
    clean_lines = [
        line for line in lines
        if line.startswith("#")
        or line.strip() == ""
        or line.count("\t") == 9
    ]
    clean_data = "\n".join(clean_lines) + "\n"

    # --- Parse sentences ---
    sentences = parse(clean_data, field_parsers=custom_field_parsers)

    for sent in sentences:
        sentence_obj = mptf.Sentence(
            sent,
            source_filename=filename
        )
        sentence_obj.metadata["source_filename"] = filename

        # → always add to the full corpus
        my_corpus.append(sentence_obj)

        # → add to syntactic corpus ONLY if deprel exists
        if any(tok.get("deprel") not in (None, "_") for tok in sent):
            syntactically_annotated_corpus.append(sentence_obj)

# ======================================================
# CONFIRMATION
# ======================================================

print("✔ Corpora loaded successfully.")
print(f"  my_corpus (all texts):")
print(f"    texts:     {len(set(s.metadata['source_filename'] for s in my_corpus))}")
print(f"    sentences: {len(my_corpus)}")

print(f"\n  syntactically_annotated_corpus:")
print(f"    texts:     {len(set(s.metadata['source_filename'] for s in syntactically_annotated_corpus))}")
print(f"    sentences: {len(syntactically_annotated_corpus)}")



--- Loading corpora ---
✔ Corpora loaded successfully.
  my_corpus (all texts):
    texts:     40
    sentences: 31191

  syntactically_annotated_corpus:
    texts:     25
    sentences: 4615


Globally:

percentage = deprel count ÷ total number of dependency-annotated tokens in the corpus

Per text:

percentage = deprel count ÷ total number of dependency-annotated tokens in that text

In [10]:
from collections import Counter, defaultdict

# ======================================================
# INITIALIZE COUNTERS
# ======================================================

# Global
global_deprel_counts = Counter()
global_total_tokens = 0

# Per text
per_text_deprel = defaultdict(Counter)
per_text_total_tokens = defaultdict(int)

# ======================================================
# ITERATE OVER SYNTACTIC CORPUS
# ======================================================

for sentence in syntactically_annotated_corpus:
    text_id = sentence.metadata.get("source_filename", "unknown")

    for token in sentence.get_tokens():

        # Skip empty or malformed tokens
        if not token.deprel:
            continue

        if token.form in {"_", "__"}:
            continue

        # If deprel contains '|', keep only the second part
        deprel = token.deprel.split("|")[-1]

        # Global
        global_deprel_counts[deprel] += 1
        global_total_tokens += 1

        # Per text
        per_text_deprel[text_id][deprel] += 1
        per_text_total_tokens[text_id] += 1

# ======================================================
# COMPUTE RATIOS
# ======================================================

# Global ratios
global_deprel_ratios = {
    dep: count / global_total_tokens * 100
    for dep, count in global_deprel_counts.items()
}

# Per-text ratios
per_text_deprel_ratios = defaultdict(dict)

for text_id, counts in per_text_deprel.items():
    total = per_text_total_tokens[text_id]
    for dep, count in counts.items():
        per_text_deprel_ratios[text_id][dep] = (
            count / total * 100 if total > 0 else 0
        )

# ======================================================
# DISPLAY RESULTS
# ======================================================

print("=== Global DEPREL Statistics ===")

for dep, count in global_deprel_counts.most_common():
    percent = (count / global_total_tokens * 100) if global_total_tokens > 0 else 0
    print(f"{dep:<15} {count:>8}  ({percent:6.2f}%)")

print("\n=== Per-Text DEPREL Statistics ===")

for text_id, dep_counts in sorted(per_text_deprel.items()):
    total = per_text_total_tokens[text_id]

    print(f"\nText: {text_id}")
    print(f"{'DEPREL':<15} {'Count':>8}  {'Percent':>9}")
    print("-" * 36)

    for dep, count in dep_counts.most_common():
        percent = (count / total * 100) if total > 0 else 0
        print(f"{dep:<15} {count:>8}  ({percent:6.2f}%)")


=== Global DEPREL Statistics ===
case                7026  (  9.43%)
det                 6776  (  9.10%)
obl                 5869  (  7.88%)
conj                5643  (  7.58%)
nmod                5549  (  7.45%)
cc                  5514  (  7.40%)
advmod              5319  (  7.14%)
nsubj               5018  (  6.74%)
root                4505  (  6.05%)
mark                3623  (  4.86%)
obj                 2372  (  3.18%)
advcl               2070  (  2.78%)
amod                2039  (  2.74%)
punct               1719  (  2.31%)
_                   1278  (  1.72%)
acl:relcl           1238  (  1.66%)
xcomp               1146  (  1.54%)
nummod               906  (  1.22%)
cop                  607  (  0.81%)
aux                  588  (  0.79%)
ccomp                567  (  0.76%)
appos                463  (  0.62%)
parataxis            458  (  0.61%)
reparandum           437  (  0.59%)
obl:lmod             327  (  0.44%)
orphan               316  (  0.42%)
dep                  314  (  0.

In [18]:
from collections import defaultdict, Counter

# ======================================================
# SETTINGS
# ======================================================

TARGET_DEPRELS = {"aux", "cop", "xcomp"}

# ======================================================
# DATA STRUCTURES
# ======================================================

# deprel -> lemma -> total count
deprel_lemma_counts = defaultdict(Counter)

# deprel -> lemma -> feat_key -> count
deprel_lemma_feat_key_counts = defaultdict(
    lambda: defaultdict(Counter)
)

# deprel -> lemma -> feat_key -> Counter(values)
deprel_lemma_feat_value_counts = defaultdict(
    lambda: defaultdict(lambda: defaultdict(Counter))
)

# ======================================================
# ITERATE OVER SYNTACTIC CORPUS
# ======================================================

for sentence in syntactically_annotated_corpus:
    for token in sentence.get_tokens():

        # Basic sanity checks
        if not token.deprel or not token.lemma:
            continue

        if token.form in {"_", "__"}:
            continue

        # If pipe-separated, keep SECOND deprel
        deprel = token.deprel.split("|")[-1]
        # Normalize LEMMA: keep FIRST if pipe-separated
    

        if deprel not in TARGET_DEPRELS:
            continue

        lemma = token.lemma.split("|")[0]
        # Count lemma per deprel
        deprel_lemma_counts[deprel][lemma] += 1

        feats = token.feats or {}

        for feat_key, feat_value in feats.items():
            deprel_lemma_feat_key_counts[deprel][lemma][feat_key] += 1
            deprel_lemma_feat_value_counts[deprel][lemma][feat_key][feat_value] += 1
print("=== DEPREL → LEMMA → FEATS STATISTICS ===")

for deprel in sorted(deprel_lemma_counts):
    print(f"\n==============================")
    print(f"DEPREL: {deprel}")
    print("==============================")

    for lemma, total in deprel_lemma_counts[deprel].most_common():
        print(f"\nLemma: {lemma}")
        print(f"  Total tokens: {total}")

        feat_keys = deprel_lemma_feat_key_counts[deprel][lemma]

        if not feat_keys:
            print("  FEATS: none")
            continue

        print("  FEATS:")

        for feat_key, key_count in feat_keys.items():
            print(f"    {feat_key}: {key_count}")

            for value, value_count in (
                deprel_lemma_feat_value_counts[deprel][lemma][feat_key].most_common()
            ):
                print(f"      {value}: {value_count}")


=== DEPREL → LEMMA → FEATS STATISTICS ===

DEPREL: aux

Lemma: ēstādan
  Total tokens: 299
  FEATS:
    Mood: 298
      Ind: 297
      Opt: 1
    Number: 298
      Sing: 284
      Plur: 14
    Person: 298
      3: 295
      1: 3
    Subcat: 299
      Intr: 299
    Tense: 299
      Pres: 268
      Past: 31
    VerbForm: 299
      Fin: 296
      Inf: 2
      Part: 1
    VerbType: 293
      Cop: 293
    Typo: 8
      Yes: 8

Lemma: h-
  Total tokens: 246
  FEATS:
    Mood: 246
      Ind: 193
      Opt: 35
      Sub: 18
    Number: 246
      Plur: 155
      Sing: 91
    Person: 246
      3: 201
      2: 26
      1: 19
    Subcat: 183
      Intr: 183
    Tense: 246
      Pres: 246
    VerbForm: 245
      Fin: 245
    VerbType: 222
      Cop: 222
    Polarity: 1
      Neg: 1
    Typo: 2
      Yes: 2

Lemma: būdan
  Total tokens: 43
  FEATS:
    Mood: 41
      Ind: 39
      Sub: 2
    Number: 41
      Sing: 40
      Plur: 1
    Person: 41
      3: 41
    Subcat: 43
      Intr: 43
    Tense: 4

In [19]:
from collections import defaultdict, Counter

# ======================================================
# TARGET DEPRELS
# ======================================================

TARGET_DEPRELS = {"aux", "cop", "xcomp"}

# ======================================================
# DATA STRUCTURE
# ======================================================
# stats[deprel][lemma] = {
#     "count": int,
#     "feats": {
#         feat_key: Counter({value: freq})
#     }
# }

stats = defaultdict(
    lambda: defaultdict(
        lambda: {
            "count": 0,
            "feats": defaultdict(Counter)
        }
    )
)

# ======================================================
# ITERATE OVER SYNTACTIC CORPUS
# ======================================================

for sentence in syntactically_annotated_corpus:

    # Build token index for head lookup
    tokens_by_id = {tok.id: tok for tok in sentence.get_tokens()}

    for token in sentence.get_tokens():

        # --------------------------
        # BASIC FILTERS
        # --------------------------
        if not token.deprel or not token.lemma:
            continue

        if token.form in {"_", "__"}:
            continue

        # --------------------------
        # NORMALIZE DEPREL
        # (take second if pipe-separated)
        # --------------------------
        deprel = token.deprel.split("|")[-1]

        if deprel not in TARGET_DEPRELS:
            continue

        # --------------------------
        # NORMALIZE LEMMA
        # (take first if pipe-separated)
        # --------------------------
        lemma = token.lemma.split("|")[0]

        if not lemma or lemma in {"_", "__"}:
            continue

        # --------------------------
        # SPECIAL RULE FOR XCOMP
        # HEAD must have VerbType=Mod
        # --------------------------
        if deprel == "xcomp":

            head = tokens_by_id.get(token.head)

            if not head or not head.feats:
                continue

            if head.feats.get("VerbType") != "Mod":
                continue

        # ==================================================
        # COUNTING
        # ==================================================

        entry = stats[deprel][lemma]
        entry["count"] += 1

        if token.feats:
            for feat_key, feat_val in token.feats.items():
                entry["feats"][feat_key][feat_val] += 1

# ======================================================
# DISPLAY RESULTS
# ======================================================

for deprel in sorted(stats):

    print(f"\n{'='*60}")
    print(f"DEPREL: {deprel}")
    print(f"{'='*60}")

    for lemma, data in sorted(
        stats[deprel].items(),
        key=lambda x: x[1]["count"],
        reverse=True
    ):

        print(f"\nLemma: {lemma}")
        print(f"  Total occurrences: {data['count']}")

        if not data["feats"]:
            print("  FEATS: none")
            continue

        print("  FEATS:")
        for feat_key, values in data["feats"].items():
            print(f"    {feat_key}:")
            for val, cnt in values.most_common():
                print(f"      {val}: {cnt}")



DEPREL: aux

Lemma: ēstādan
  Total occurrences: 299
  FEATS:
    Mood:
      Ind: 297
      Opt: 1
    Number:
      Sing: 284
      Plur: 14
    Person:
      3: 295
      1: 3
    Subcat:
      Intr: 299
    Tense:
      Pres: 268
      Past: 31
    VerbForm:
      Fin: 296
      Inf: 2
      Part: 1
    VerbType:
      Cop: 293
    Typo:
      Yes: 8

Lemma: h-
  Total occurrences: 246
  FEATS:
    Mood:
      Ind: 193
      Opt: 35
      Sub: 18
    Number:
      Plur: 155
      Sing: 91
    Person:
      3: 201
      2: 26
      1: 19
    Subcat:
      Intr: 183
    Tense:
      Pres: 246
    VerbForm:
      Fin: 245
    VerbType:
      Cop: 222
    Polarity:
      Neg: 1
    Typo:
      Yes: 2

Lemma: būdan
  Total occurrences: 43
  FEATS:
    Mood:
      Ind: 39
      Sub: 2
    Number:
      Sing: 40
      Plur: 1
    Person:
      3: 41
    Subcat:
      Intr: 43
    Tense:
      Past: 22
      Pres: 20
    VerbForm:
      Fin: 41
      Inf: 1
      Part: 1
    VerbType:
   

In [46]:
from collections import defaultdict, Counter

# ======================================================
# DATA STRUCTURE FOR AUXILIARIES
# ======================================================
# aux_stats[lemma] = {
#     "count": int,
#     "feats": {feat_key: Counter({feat_val: freq})}
# }
aux_stats = defaultdict(lambda: {"count": 0, "feats": defaultdict(Counter)})

# ======================================================
# ITERATE OVER SYNTACTICALLY ANNOTATED CORPUS
# ======================================================
for sentence in syntactically_annotated_corpus:
    for token in sentence.get_tokens():

        # --------------------------
        # FILTERS
        # --------------------------
        if not token.deprel or not token.lemma:
            continue
        if token.form in {"_", "__"}:
            continue
        # Only auxiliaries
        if token.deprel.split("|")[-1] != "aux":
            continue

        # Normalize lemma (take first pipe variant without spaces)
        lemma_candidates = token.lemma.split("|")
        lemma = next((l for l in lemma_candidates if " " not in l), lemma_candidates[0])

        if not lemma or lemma in {"_", "__"}:
            continue

        # --------------------------
        # COUNTING
        # --------------------------
        entry = aux_stats[lemma]
        entry["count"] += 1

        if token.feats:
            for feat_key, feat_val in token.feats.items():
                entry["feats"][feat_key][feat_val] += 1

# ======================================================
# DISPLAY RESULTS
# ======================================================
print("\n" + "="*60)
print("AUXILIARY LEMMAS AND THEIR FEATS")
print("="*60)

for lemma, data in sorted(aux_stats.items(), key=lambda x: x[1]["count"], reverse=True):
    print(f"\nLemma: {lemma}")
    print(f"  Total occurrences: {data['count']}")

    if not data["feats"]:
        print("  FEATS: none")
        continue

    print("  FEATS:")
    for feat_key, val_counter in data["feats"].items():
        print(f"    {feat_key}:")
        for val, cnt in val_counter.most_common():
            print(f"      {val}: {cnt}")



AUXILIARY LEMMAS AND THEIR FEATS

Lemma: ēstādan
  Total occurrences: 299
  FEATS:
    Mood:
      Ind: 297
      Opt: 1
    Number:
      Sing: 284
      Plur: 14
    Person:
      3: 295
      1: 3
    Subcat:
      Intr: 299
    Tense:
      Pres: 268
      Past: 31
    VerbForm:
      Fin: 296
      Inf: 2
      Part: 1
    VerbType:
      Cop: 293
    Typo:
      Yes: 8

Lemma: h-
  Total occurrences: 246
  FEATS:
    Mood:
      Ind: 193
      Opt: 35
      Sub: 18
    Number:
      Plur: 155
      Sing: 91
    Person:
      3: 201
      2: 26
      1: 19
    Subcat:
      Intr: 183
    Tense:
      Pres: 246
    VerbForm:
      Fin: 245
    VerbType:
      Cop: 222
    Polarity:
      Neg: 1
    Typo:
      Yes: 2

Lemma: būdan
  Total occurrences: 43
  FEATS:
    Mood:
      Ind: 39
      Sub: 2
    Number:
      Sing: 40
      Plur: 1
    Person:
      3: 41
    Subcat:
      Intr: 43
    Tense:
      Past: 22
      Pres: 20
    VerbForm:
      Fin: 41
      Inf: 1
      Part

In [47]:
from collections import defaultdict, Counter

# ------------------------------
# Helper function to normalize lemma
# ------------------------------
def select_lemma(lemma_str):
    candidates = lemma_str.split("|")
    for cand in candidates:
        if " " not in cand:
            return cand
    return candidates[0]

# ------------------------------
# Initialize data structure
# ------------------------------
# aux_phrases[aux_lemma] = Counter({phrase_str: count})
aux_phrases = defaultdict(Counter)

# ------------------------------
# Iterate over syntactic corpus
# ------------------------------
for sentence in syntactically_annotated_corpus:
    tokens = sentence.get_tokens()
    token_by_id = {str(tok.id): tok for tok in tokens if tok.id is not None}

    for token in tokens:
        # --------------------------
        # Basic filters
        # --------------------------
        if not token.lemma or not token.feats or token.form in {"_", "__"}:
            continue

        # Only auxiliaries
        if token.deprel.split("|")[-1] != "aux":
            continue

        aux_lemma = select_lemma(token.lemma)

        # --------------------------
        # Get head token of the auxiliary
        # --------------------------
        head_token = token_by_id.get(str(token.head))
        if not head_token or not head_token.lemma:
            continue

        head_lemma = select_lemma(head_token.lemma)

        # --------------------------
        # Get particle dependents of the head
        # --------------------------
        particle_set = {"bē", "be", "nē", "ham", "hamēw"}
        head_dependents = [
            dep for dep in tokens
            if str(dep.head) == str(head_token.id) and select_lemma(dep.lemma) in particle_set
        ]

        # --------------------------
        # Build phrase: particles + head + auxiliary
        # --------------------------
        phrase_tokens = head_dependents + [head_token, token]  # particles, head, aux
        phrase_tokens_sorted = sorted(phrase_tokens, key=lambda x: x.id)
        phrase_lemmas = [select_lemma(t.lemma) for t in phrase_tokens_sorted]
        phrase_str = " ".join(phrase_lemmas)

        aux_phrases[aux_lemma][phrase_str] += 1

# ------------------------------
# Display results
# ------------------------------
print("\n" + "="*60)
print("AUXILIARY PHRASES WITH PARTICLES")
print("="*60)

for aux_lemma, phrases in sorted(aux_phrases.items()):
    print(f"\nAuxiliary: {aux_lemma}")
    for phrase, count in phrases.most_common():
        print(f"  {phrase}: {count}")



AUXILIARY PHRASES WITH PARTICLES

Auxiliary: būdan
  nē tuwān būdan: 6
  kirdan būdan: 4
  dādan būdan: 4
  šnāyēnīdan būdan: 4
  guftan būdan: 3
  stāyīdār būdan: 1
  ēdōn būdan: 1
  dāštan būdan: 1
  wirāstan būdan: 1
  madan būdan: 1
  ōzadan būdan: 1
  tuwān būdan: 1
  brēhēnīdan būdan: 1
  kāmag būdan: 1
  hamkōxšīdan būdan: 1
  ǰastan būdan: 1
  bē tazīdan būdan: 1
  paydāg būdan: 1
  wisāndan būdan: 1
  xwardan būdan: 1
  murnǰēnīdan būdan: 1
  bē dādan būdan: 1
  be griftan būdan: 1
  abzāyišn būdan: 1
  āmārīhā būdan: 1
  paydāgēnīdan būdan: 1
  būdan būdan: 1

Auxiliary: h-
  būdan h-: 20
  madan h-: 13
  kirdan h-: 8
  dādan h-: 7
  rustan h-: 7
  ēstādan h-: 6
  šudan h-: 6
  raftan1 h-: 5
  waštan h-: 4
  dwāristan h-: 4
  ǰastan h-: 4
  guftan h-: 4
  āmadan h-: 3
  gumārdan h-: 3
  waxšīdan h-: 3
  axwāstār h-: 3
  nē guhrādan h-: 3
  māndan h-: 2
  nē šāyistan h-: 2
  xwāstan h-: 2
  gumēxtan h-: 2
  kōxšīdan h-: 2
  nē būdan h-: 2
  bastan h-: 2
  tazīdan h-: 2
  bē t

In [20]:
#investigation of Lightverbs
from collections import defaultdict, Counter

# ======================================================
# DATA STRUCTURE
# ======================================================
# light_verbs[lemma] = {
#     "count": int,
#     "feats": {
#         feat_key: Counter({value: freq})
#     }
# }

light_verbs = defaultdict(
    lambda: {
        "count": 0,
        "feats": defaultdict(Counter)
    }
)

# ======================================================
# ITERATE OVER SYNTACTIC CORPUS
# ======================================================

for sentence in syntactically_annotated_corpus:

    for token in sentence.get_tokens():

        # --------------------------
        # BASIC FILTERS
        # --------------------------
        if not token.lemma or not token.feats:
            continue

        if token.form in {"_", "__"}:
            continue

        # --------------------------
        # CHECK VerbType=Light
        # --------------------------
        if token.feats.get("VerbType") != "Light":
            continue

        # --------------------------
        # NORMALIZE LEMMA
        # (take first if pipe-separated)
        # --------------------------
        lemma = token.lemma.split("|")[0]

        if not lemma or lemma in {"_", "__"}:
            continue

        # ==================================================
        # COUNTING
        # ==================================================

        entry = light_verbs[lemma]
        entry["count"] += 1

        for feat_key, feat_val in token.feats.items():
            entry["feats"][feat_key][feat_val] += 1

# ======================================================
# DISPLAY RESULTS
# ======================================================

print("\n" + "=" * 60)
print("VERBS WITH VerbType=Light")
print("=" * 60)

for lemma, data in sorted(
    light_verbs.items(),
    key=lambda x: x[1]["count"],
    reverse=True
):

    print(f"\nLemma: {lemma}")
    print(f"  Total occurrences: {data['count']}")

    if not data["feats"]:
        print("  FEATS: none")
        continue

    print("  FEATS:")
    for feat_key, values in data["feats"].items():
        print(f"    {feat_key}:")
        for val, cnt in values.most_common():
            print(f"      {val}: {cnt}")



VERBS WITH VerbType=Light

Lemma: kirdan
  Total occurrences: 141
  FEATS:
    Mood:
      Ind: 92
      Imp: 8
      Nec: 5
    Subcat:
      Tran: 139
      Intr: 2
    VerbForm:
      Fin: 101
      Inf: 30
      Part: 8
      Vnoun: 2
    VerbType:
      Light: 141
    Number:
      Sing: 87
      Plur: 14
    Person:
      3: 89
      2: 9
      1: 3
    Tense:
      Pres: 57
      Past: 52
    Typo:
      Yes: 7

Lemma: burdan
  Total occurrences: 21
  FEATS:
    Mood:
      Ind: 13
      Imp: 3
      Nec: 1
    Number:
      Sing: 13
      Plur: 3
    Person:
      3: 11
      2: 4
      1: 1
    Subcat:
      Tran: 21
    Tense:
      Pres: 12
      Past: 4
    VerbForm:
      Fin: 16
      Inf: 4
      Vnoun: 1
    VerbType:
      Light: 21
    Animacy:
      Inan: 1
    Typo:
      Yes: 1

Lemma: būdan
  Total occurrences: 20
  FEATS:
    Mood:
      Ind: 18
      Imp: 2
    Number:
      Sing: 19
      Plur: 1
    Person:
      3: 18
      2: 2
    Subcat:
      Intr: 20
  

In [43]:
from collections import defaultdict, Counter

# ======================================================
# DATA STRUCTURE
# ======================================================
# lightverb_lvc[light_lemma] = {
#     "total": int,
#     "lvc_dependents": Counter({dependent_lemma: count})
# }
lightverb_lvc = defaultdict(lambda: {"total": 0, "lvc_dependents": Counter()})

def select_lemma(lemma_str):
    """
    Given a lemma that might be pipe-separated, return the first lemma
    that does NOT contain a space. If all contain spaces, fall back to the first.
    """
    candidates = lemma_str.split("|")
    for cand in candidates:
        if " " not in cand:
            return cand
    return candidates[0]  # fallback if all contain spaces


# ======================================================
# ITERATE OVER SYNTACTIC CORPUS
# ======================================================
for sentence in syntactically_annotated_corpus:
    tokens = sentence.get_tokens()

    # Build ID → token map for easy lookup
    token_by_id = {str(tok.id): tok for tok in tokens if tok.id is not None}

    for token in tokens:

        # --------------------------
        # BASIC FILTERS
        # --------------------------
        if not token.lemma or not token.feats:
            continue
        if token.form in {"_", "__"}:
            continue

        # --------------------------
        # CHECK VerbType=Light
        # --------------------------
        if token.feats.get("VerbType") != "Light":
            continue

        # --------------------------
        # NORMALIZE LIGHT VERB LEMMA
        # --------------------------
        light_lemma = select_lemma(token.lemma)
        if not light_lemma:
            continue

        # --------------------------
        # FIND DEPENDENTS WITH deprel=compound:lvc
        # --------------------------
        for dep in tokens:
            if str(dep.head) == str(token.id) and dep.deprel == "compound:lvc":
                dep_lemma = select_lemma(dep.lemma)
                lightverb_lvc[light_lemma]["lvc_dependents"][dep_lemma] += 1
                lightverb_lvc[light_lemma]["total"] += 1

# ======================================================
# DISPLAY RESULTS
# ======================================================
print("\n" + "="*60)
print("Light verbs and their Preverbs")
print("="*60)

for light_lemma, data in sorted(
    lightverb_lvc.items(),
    key=lambda x: x[1]["total"],
    reverse=True
):
    print(f"\nLight verb: {light_lemma}")
    print(f"  Total LVC occurrences: {data['total']}")
    print("  LVC dependents:")
    for dep_lemma, count in data["lvc_dependents"].most_common():
        print(f"    {dep_lemma}: {count}")



Light verbs and their Preverbs

Light verb: kirdan
  Total LVC occurrences: 87
  LVC dependents:
    passox: 21
    šōy1: 9
    zan: 7
    warm: 5
    pāk: 3
    pānagīh: 3
    mehmān: 2
    uzēnag: 2
    hammōzišn: 2
    wāng: 2
    hučašmīh: 1
    gāh1: 1
    bahr: 1
    āwēnišn: 1
    paymān: 1
    šēwan: 1
    ayārōmandīh: 1
    griftār: 1
    tar: 1
    ayād: 1
    stāyišn: 1
    aspinǰ: 1
    paristišn: 1
    nigān: 1
    rawāg: 1
    paydāg: 1
    abzōn: 1
    xwadāyīh: 1
    gugān: 1
    wardānāg: 1
    nišānīdīg: 1
    wizend: 1
    pādifrāh: 1
    graw: 1
    druyist: 1
    hangōšīdag: 1
    xwurdagčīnīh: 1
    tēzbrīnīh: 1
    passand: 1
    abarpād: 1
    rōšn: 1

Light verb: dāštan
  Total LVC occurrences: 11
  LVC dependents:
    nigāh: 4
    mehmān: 1
    bēš: 1
    spās: 1
    burd: 1
    kār1: 1
    mayān: 1
    nihān: 1

Light verb: pādan1
  Total LVC occurrences: 2
  LVC dependents:
    pās: 2

Light verb: guftan
  Total LVC occurrences: 2
  LVC dependents:
    mih:

In [ ]:
# Give light verb lemma and get sentence ids where it occurs

In [44]:
from collections import defaultdict

# ======================================================
# INPUT: light verb lemma AND dependent lemma (compound:lvc)
# ======================================================

input_light_lemma = "kirdan"     # Light verb lemma
input_dep_lemma   = "ayārōmandīh"    # Dependent lemma you want to match

# ======================================================
# COLLECT SENTENCE IDS
# ======================================================

sentences_with_light_and_dep = set()

for sentence in syntactically_annotated_corpus:

    sentence_id = sentence.metadata.get("SENTENCE ID", "UNKNOWN_SENTENCE_ID")
    tokens = sentence.get_tokens()

    # Build ID → token map for easy lookup
    token_by_id = {str(tok.id): tok for tok in tokens if tok.id is not None}

    for token in tokens:

        # --------------------------
        # BASIC FILTERS
        # --------------------------
        if not token.lemma or not token.feats:
            continue
        if token.form in {"_", "__"}:
            continue

        # --------------------------
        # ONLY LIGHT VERBS
        # --------------------------
        if token.feats.get("VerbType") != "Light":
            continue

        # --------------------------
        # NORMALIZE LIGHT VERB LEMMA
        # --------------------------
        lemmata = token.lemma.split("|")
        light_lemma = lemmata[1] if len(lemmata) > 1 else lemmata[0]
        if light_lemma != input_light_lemma:
            continue

        # --------------------------
        # CHECK DEPENDENTS WITH deprel=compound:lvc
        # --------------------------
        for dep in tokens:
            if str(dep.head) == str(token.id) and dep.deprel == "compound:lvc":
                dep_lemmata = dep.lemma.split("|")
                dep_lemma_norm = dep_lemmata[1] if len(dep_lemmata) > 1 else dep_lemmata[0]

                if dep_lemma_norm == input_dep_lemma:
                    sentences_with_light_and_dep.add(sentence_id)

# ======================================================
# OUTPUT
# ======================================================

print(f"\nSentences where light verb '{input_light_lemma}' has LVC dependent '{input_dep_lemma}':\n")

for sid in sorted(sentences_with_light_and_dep):
    print(sid)

print(f"\nTotal sentences: {len(sentences_with_light_and_dep)}")



Sentences where light verb 'kirdan' has LVC dependent 'ayārōmandīh':

DMX-L19 s40	_	_	_	_	_	_	_	_	_

Total sentences: 1


In [45]:
# light verb constructions together with the preverbs
from collections import defaultdict, Counter

# ------------------------------
# Helper function to normalize lemma
# ------------------------------
def select_lemma(lemma_str):
    candidates = lemma_str.split("|")
    for cand in candidates:
        if " " not in cand:
            return cand
    return candidates[0]

# ------------------------------
# Initialize data structure
# ------------------------------
# lightverb_phrases[light_lemma] = Counter({phrase_str: count})
lightverb_phrases = defaultdict(Counter)

# ------------------------------
# Iterate over corpus
# ------------------------------
for sentence in syntactically_annotated_corpus:
    tokens = sentence.get_tokens()
    token_by_id = {str(tok.id): tok for tok in tokens if tok.id is not None}

    for token in tokens:
        # Basic filters
        if not token.lemma or not token.feats or token.form in {"_", "__"}:
            continue
        if token.feats.get("VerbType") != "Light":
            continue

        # Normalize light verb lemma
        light_lemma = select_lemma(token.lemma)

        # Find compound:lvc dependents
        lvc_dependents = [dep for dep in tokens if str(dep.head) == str(token.id) and dep.deprel == "compound:lvc"]
        if not lvc_dependents:
            continue  # no LVC, skip

        # Find particles: bē/be, nē, ham/hamēw
        particle_set = {"bē", "be", "nē", "ham", "hamēw"}
        particles = [dep for dep in tokens if str(dep.head) == str(token.id) and select_lemma(dep.lemma) in particle_set]

        # Combine LVC and particles, sort by token.id to get correct order
        for lvc in lvc_dependents:
            phrase_tokens = particles + [lvc, token]  # particles, LVC, light verb
            # sort by token.id
            phrase_tokens_sorted = sorted(phrase_tokens, key=lambda x: x.id)
            # convert to lemmas
            phrase_lemmas = [select_lemma(t.lemma) for t in phrase_tokens_sorted]
            phrase_str = " ".join(phrase_lemmas)
            lightverb_phrases[light_lemma][phrase_str] += 1

# ------------------------------
# Display results
# ------------------------------
for light_lemma, phrases in sorted(lightverb_phrases.items()):
    print(f"\nLight verb: {light_lemma}")
    for phrase, count in phrases.most_common():
        print(f"  {phrase}: {count}")



Light verb: burdan
  namāz burdan: 1
  čōb burdan: 1

Light verb: būdan
  bun būdan: 1

Light verb: dāštan
  nigāh dāštan: 4
  mehmān dāštan: 1
  bēš dāštan: 1
  spās dāštan: 1
  burd dāštan: 1
  kār1 dāštan: 1
  mayān dāštan: 1
  nihān dāštan: 1

Light verb: framūdan
  kār1 framūdan: 1

Light verb: guftan
  mih guftan: 1
  nām guftan: 1

Light verb: kirdan
  passox kirdan: 21
  šōy1 kirdan: 9
  zan kirdan: 5
  warm kirdan: 3
  pāk kirdan: 3
  pānagīh kirdan: 3
  mehmān be kirdan: 2
  uzēnag kirdan: 2
  hammōzišn kirdan: 2
  wāng kirdan: 2
  warm be kirdan: 2
  hučašmīh kirdan: 1
  gāh1 kirdan: 1
  bahr kirdan: 1
  āwēnišn kirdan: 1
  paymān kirdan: 1
  šēwan nē kirdan: 1
  ayārōmandīh kirdan: 1
  griftār kirdan: 1
  tar kirdan: 1
  ayād kirdan: 1
  stāyišn kirdan: 1
  aspinǰ nē kirdan: 1
  paristišn kirdan: 1
  nigān kirdan: 1
  rawāg kirdan: 1
  nē paydāg kirdan: 1
  abzōn kirdan: 1
  xwadāyīh kirdan: 1
  gugān kirdan: 1
  wardānāg kirdan: 1
  hamēw zan kirdan: 1
  nišānīdīg kirdan: